# Multilingual RAG — Indian Government Policy Q&A

Answer questions about Indian government policies in English, Hindi, or Tamil.
The pipeline detects the query language, retrieves relevant documents using
cross-lingual semantic search, generates an answer with `sarvam-m`, then
translates the answer back to the user's language.

## Pipeline

```
User query (any supported language)
        |
        v
detect_query_language()    <- Sarvam identify_language API
        |
        v
retrieve_documents()       <- sentence-transformers + FAISS
        |  query embedded directly in its original language
        |  paraphrase-multilingual-mpnet-base-v2
        v
generate_answer()          <- sarvam-m (chat.completions)
        |  context: content_english from retrieved docs
        |  query translated to English for the LLM prompt
        v
translate_text()           <- Sarvam translate API
        |  skipped when detected language is en-IN
        v
Answer in the user's language
```

## Supported Languages

| Code | Language |
|:-----|:---------|
| en-IN | English (India) |
| hi-IN | Hindi |
| ta-IN | Tamil |

In [ ]:
%pip install sarvamai>=0.1.26 sentence-transformers>=2.2.2 faiss-cpu>=1.7.4 python-dotenv>=1.0.0 pandas>=1.5.0 numpy>=1.24.0

## Setup and API Key

Load environment variables and initialise the Sarvam AI client.
Set `SARVAM_API_KEY` in a `.env` file copied from `.env.example`.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sarvamai import SarvamAI
from sentence_transformers import SentenceTransformer

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY", "")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "SARVAM_API_KEY is not set. "
        "Copy .env.example to .env and paste your key."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
print("Sarvam AI client initialised.")

## Load Multilingual Documents

Load the sample corpus from `sample_data/sample_docs.json`.
Each document has a `content` field in its original language for embedding
and a `content_english` field used as context for the LLM.

In [ ]:
def load_documents(json_path: str) -> pd.DataFrame:
    """Load sample documents from a JSON file into a DataFrame.

    Args:
        json_path: Path to the JSON file containing a list of document dicts.

    Returns:
        DataFrame with one row per document.
    """
    path = Path(json_path)
    with path.open(encoding="utf-8") as fh:
        docs = json.load(fh)
    return pd.DataFrame(docs)

In [ ]:
DOCS_PATH = "sample_data/sample_docs.json"
df = load_documents(DOCS_PATH)

print(f"Loaded {len(df)} documents")
print(f"Languages : {sorted(df['language'].unique())}")
print(f"Topics    : {sorted(df['topic'].unique())}")
df[["id", "language", "topic", "title"]]

## Build Embedding Index

Embed every document using a multilingual sentence-transformer model.
The model maps semantically equivalent text across languages to nearby
vectors, enabling cross-lingual retrieval without pre-translating the query.

Model: `paraphrase-multilingual-mpnet-base-v2`
- Dimensions: 768
- Languages: 50+
- Download size: ~420 MB on first run (cached locally after that)

In [ ]:
def embed_documents(
    df: pd.DataFrame,
    model_name: str = "paraphrase-multilingual-mpnet-base-v2",
) -> tuple[Any, np.ndarray]:
    """Embed all documents using a multilingual sentence-transformer model.

    Documents are embedded from their original-language `content` field.
    Embeddings are L2-normalised so that dot product equals cosine similarity.

    Args:
        df: DataFrame containing a `content` column.
        model_name: HuggingFace model identifier for SentenceTransformer.

    Returns:
        Tuple of (loaded model, float32 embeddings array of shape (n, dim)).
    """
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        df["content"].tolist(),
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return model, np.array(embeddings, dtype="float32")

In [ ]:
def build_faiss_index(embeddings: np.ndarray) -> Any:
    """Build a FAISS IndexFlatIP index from L2-normalised embeddings.

    IndexFlatIP computes inner product, which equals cosine similarity
    when vectors are L2-normalised. No training step is required.

    Args:
        embeddings: float32 ndarray of shape (n, dim), L2-normalised.

    Returns:
        Populated FAISS IndexFlatIP index ready for search.
    """
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

In [ ]:
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"

embedding_model, doc_embeddings = embed_documents(df, EMBEDDING_MODEL_NAME)
faiss_index = build_faiss_index(doc_embeddings)

print(f"Model          : {EMBEDDING_MODEL_NAME}")
print(f"Embedding dims : {doc_embeddings.shape[1]}")
print(f"Docs indexed   : {faiss_index.ntotal}")

## Query Pipeline

Functions for language detection, cross-lingual retrieval, and translation.

In [ ]:
def detect_query_language(query: str, sarvam_client: SarvamAI) -> str:
    """Detect the BCP-47 language code of a query using Sarvam identify_language.

    Falls back to "en-IN" if the API call fails, so the pipeline degrades
    gracefully rather than raising an exception during retrieval.

    Args:
        query: Input text in any supported Indian language.
        sarvam_client: Initialised SarvamAI client.

    Returns:
        BCP-47 language code string, e.g. "hi-IN" or "ta-IN".
    """
    try:
        response = sarvam_client.text.identify_language(input=query)
        return response.language_code
    except Exception:
        return "en-IN"

In [ ]:
def retrieve_documents(
    query: str,
    embedding_model: Any,
    index: Any,
    corpus_df: pd.DataFrame,
    top_k: int = 3,
) -> pd.DataFrame:
    """Retrieve the top-k most semantically similar documents to a query.

    The query is embedded directly in its original language — no pre-translation.
    faiss.normalize_L2 ensures the query vector has unit norm before the inner
    product search, matching the normalisation applied to the document embeddings.

    Args:
        query: Input text in any supported language.
        embedding_model: Loaded SentenceTransformer model.
        index: Populated FAISS IndexFlatIP index.
        corpus_df: DataFrame of documents with metadata columns.
        top_k: Number of documents to return.

    Returns:
        DataFrame of top-k documents with an added `score` column (cosine sim).
    """
    q_emb = embedding_model.encode([query], normalize_embeddings=False)
    q_emb = np.array(q_emb, dtype="float32")
    faiss.normalize_L2(q_emb)

    scores, indices = index.search(q_emb, top_k)
    results = corpus_df.iloc[indices[0]].copy()
    results["score"] = scores[0]
    return results.reset_index(drop=True)

In [ ]:
def translate_text(
    text: str,
    source_lang: str,
    target_lang: str,
    sarvam_client: SarvamAI,
) -> str:
    """Translate text between two BCP-47 language codes using Sarvam translate.

    Returns the original text unchanged if the translation API call fails,
    so the pipeline degrades gracefully.

    Args:
        text: Text to translate.
        source_lang: Source BCP-47 code, e.g. "hi-IN".
        target_lang: Target BCP-47 code, e.g. "en-IN".
        sarvam_client: Initialised SarvamAI client.

    Returns:
        Translated text, or the original text on failure.
    """
    try:
        response = sarvam_client.text.translate(
            input=text,
            source_language_code=source_lang,
            target_language_code=target_lang,
            model="sarvam-translate:v1",
        )
        return response.translated_text
    except Exception:
        return text

## Generate Answer

Pass the English-translated query and pre-stored `content_english` fields
from the retrieved documents as context to `sarvam-m`.

In [ ]:
def generate_answer(
    query_english: str,
    retrieved_docs: pd.DataFrame,
    sarvam_client: SarvamAI,
) -> str:
    """Generate an English answer using sarvam-m with retrieved document context.

    Args:
        query_english: The user query translated to English.
        retrieved_docs: DataFrame of top-k retrieved documents, each with
            a `content_english` column used as grounding context.
        sarvam_client: Initialised SarvamAI client.

    Returns:
        Answer string in English.
    """
    context = "\n\n".join(retrieved_docs["content_english"].tolist())

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant answering questions about Indian "
                "government policies. Answer using only the provided context. "
                "If the answer is not present in the context, say you do not know."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query_english}",
        },
    ]

    response = sarvam_client.chat.completions(messages=messages, model="sarvam-m")
    return response.choices[0].message.content

## Full RAG Pipeline

`run_rag_pipeline` orchestrates all steps end-to-end for any supported language.

In [ ]:
def run_rag_pipeline(
    query: str,
    sarvam_client: SarvamAI,
    embedding_model: Any,
    index: Any,
    corpus_df: pd.DataFrame,
    top_k: int = 3,
) -> dict[str, Any]:
    """Run the full multilingual RAG pipeline for a given query.

    Steps:
        1. Detect the query language with Sarvam identify_language.
        2. Embed the query directly (no pre-translation) and retrieve top-k docs.
        3. Translate the query to English for the LLM prompt.
        4. Generate an English answer with sarvam-m using retrieved context.
        5. Translate the answer back to the user's detected language.

    Translation steps 3 and 5 are skipped when the detected language is en-IN.

    Args:
        query: User query in any supported Indian language.
        sarvam_client: Initialised SarvamAI client.
        embedding_model: Loaded SentenceTransformer model.
        index: Populated FAISS IndexFlatIP index.
        corpus_df: DataFrame of documents with metadata and content columns.
        top_k: Number of documents to retrieve.

    Returns:
        Dict with keys:
            query            - original user query
            detected_language - BCP-47 code returned by identify_language
            retrieved_docs   - list of dicts with id, title, score
            answer_english   - raw English answer from sarvam-m
            answer           - final answer in the user's detected language
    """
    detected_lang = detect_query_language(query, sarvam_client)

    retrieved = retrieve_documents(query, embedding_model, index, corpus_df, top_k)

    if detected_lang != "en-IN":
        query_english = translate_text(query, detected_lang, "en-IN", sarvam_client)
    else:
        query_english = query

    answer_english = generate_answer(query_english, retrieved, sarvam_client)

    if detected_lang != "en-IN":
        answer = translate_text(answer_english, "en-IN", detected_lang, sarvam_client)
    else:
        answer = answer_english

    retrieved_docs_summary = retrieved[["id", "title", "score"]].to_dict(orient="records")

    return {
        "query": query,
        "detected_language": detected_lang,
        "retrieved_docs": retrieved_docs_summary,
        "answer_english": answer_english,
        "answer": answer,
    }

## Demo 1 — English Query

Query about PM-KISAN farmer registration in English.
Expected retrieval: `agr_001`, `agr_002`, `agr_003`.

In [ ]:
result_en = run_rag_pipeline(
    query="How do farmers register for PM-KISAN?",
    sarvam_client=client,
    embedding_model=embedding_model,
    index=faiss_index,
    corpus_df=df,
)

print("Query            :", result_en["query"])
print("Detected language:", result_en["detected_language"])
print("Retrieved docs   :")
for doc in result_en["retrieved_docs"]:
    print(f"  [{doc['id']}] {doc['title']}  (score: {doc['score']:.3f})")
print()
print("Answer:")
print(result_en["answer"])

## Demo 2 — Hindi Query

Query about the NEP 2020 curriculum structure in Hindi.
Expected retrieval: `edu_001`, `edu_002`, `edu_003`.

In [ ]:
result_hi = run_rag_pipeline(
    query="नई शिक्षा नीति में पाठ्यक्रम संरचना क्या है?",
    sarvam_client=client,
    embedding_model=embedding_model,
    index=faiss_index,
    corpus_df=df,
)

print("Query            :", result_hi["query"])
print("Detected language:", result_hi["detected_language"])
print("Retrieved docs   :")
for doc in result_hi["retrieved_docs"]:
    print(f"  [{doc['id']}] {doc['title']}  (score: {doc['score']:.3f})")
print()
print("Answer (Hindi):")
print(result_hi["answer"])
print()
print("Answer (English, before translation):")
print(result_hi["answer_english"])

## Demo 3 — Tamil Query

Query about Aadhaar enrollment in Tamil.
Expected retrieval: `did_001`, `did_002`, `did_003`.

In [ ]:
result_ta = run_rag_pipeline(
    query="ஆதார் எண் எப்படி பெறுவது?",
    sarvam_client=client,
    embedding_model=embedding_model,
    index=faiss_index,
    corpus_df=df,
)

print("Query            :", result_ta["query"])
print("Detected language:", result_ta["detected_language"])
print("Retrieved docs   :")
for doc in result_ta["retrieved_docs"]:
    print(f"  [{doc['id']}] {doc['title']}  (score: {doc['score']:.3f})")
print()
print("Answer (Tamil):")
print(result_ta["answer"])
print()
print("Answer (English, before translation):")
print(result_ta["answer_english"])

## Error Reference and Resources

### Common Errors

| Error | Cause | Fix |
|:------|:------|:----|
| `RuntimeError: SARVAM_API_KEY is not set` | Missing `.env` or empty key | Copy `.env.example` to `.env` and add your key |
| `ModuleNotFoundError: No module named 'faiss'` | FAISS not installed | `pip install faiss-cpu` |
| `sarvamai.APIStatusError: 401` | Invalid API key | Verify key at dashboard.sarvam.ai |
| `sarvamai.APIStatusError: 429` | Rate limit exceeded | Add a short delay between requests |
| `OSError: [Errno 28] No space left on device` | Model download failed | Free at least 1 GB disk space |

### Resources

- [Sarvam AI Documentation](https://docs.sarvam.ai/)
- [Sarvam API Dashboard](https://dashboard.sarvam.ai/)
- [paraphrase-multilingual-mpnet-base-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2)
- [FAISS documentation](https://faiss.ai/)